# Phase 3: Inventory Policy Optimization

This notebook optimizes inventory policies for the selected product families under the base case cost model. It constructs simulated risk-period demands, calculates heuristics (initial safety stock and EOQ), executes a two-stage grid search (coarse and fine) subject to the service level constraint (fill rate $\beta \ge 0.95$), and compares grid-search policies against neighborhood local search.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.config import Config
from src.data_loader import load_train_data, load_test_data, filter_families
from src.demand_uncertainty import calculate_ratios, center_ratios, cap_by_percentile
from src.distributions import bootstrap_sample
from src.simulation import InventorySimulator
from src.optimization import GridSearchOptimizer, LocalSearchOptimizer, calculate_eoq
from src.reporting import create_policy_comparison_table

## 1. Load Configurations and Data

In [2]:
config = Config('../configs/inventory_config.yaml')
train_df = load_train_data('../../Demand Forecasting System/data/actual_fitted_sales_on_train_final.csv', config)
test_df = load_test_data('../../Demand Forecasting System/data/actual_fitted_sales_on_test_final.csv', config)

train_df = filter_families(train_df, config.selected_families, config)
test_df = filter_families(test_df, config.selected_families, config)

## 2. Generate Simulated Demand Paths

In [3]:
col_family = config.columns['family']
col_actual = config.columns['train_actual']
col_fitted = config.columns['train_fitted']
col_forecast = config.columns['test_forecast']

simulated_weekly_demand = {}
test_forecasts = {}
historical_actuals = {}
historical_forecasts = {}

n_reps = config.n_replications
rng = np.random.default_rng(config.random_seed)

for family in config.selected_families:
    fam_df = train_df[train_df[col_family] == family]
    hist_act = fam_df[col_actual].values
    hist_fit = fam_df[col_fitted].values
    test_f = test_df[test_df[col_family] == family][col_forecast].values
    
    historical_actuals[family] = hist_act
    historical_forecasts[family] = hist_fit
    test_forecasts[family] = test_f
    
    # Generate forecast ratios
    ratios = calculate_ratios(hist_act, hist_fit, config.epsilon)
    mean_ratio = np.mean(ratios)
    ratios_centered = center_ratios(ratios, mean_ratio)
    ratios_capped = cap_by_percentile(ratios_centered, 1.0, 99.0)
    
    # Generate simulation paths
    n_weeks_test = len(test_f)
    weekly_paths = np.zeros((n_reps, n_weeks_test))
    for i in range(n_reps):
        sampled_r = bootstrap_sample(ratios_capped, size=n_weeks_test, random_state=rng)
        weekly_paths[i, :] = np.maximum(0.0, test_f * sampled_r)
        
    simulated_weekly_demand[family] = weekly_paths
    print(f"Generated {n_reps} demand paths for {family} of length {n_weeks_test} weeks.")

Generated 10000 demand paths for GROCERY I of length 3 weeks.
Generated 10000 demand paths for BEVERAGES of length 3 weeks.
Generated 10000 demand paths for CLEANING of length 3 weeks.


## 3. Run Optimization per Product Family

In [4]:
optimization_results = []

for family in config.selected_families:
    p_type = config.policies.get(family, 'SQ')
    L = config.lead_times.get(family, 1)
    R = config.review_periods.get(family, 1)
    
    H_weekly = config.holding_cost_weekly
    K = config.order_cost
    B = config.shortage_costs.get(family, 15.0)
    
    weekly_paths = simulated_weekly_demand[family]
    
    # Determine risk period tau
    tau = L if p_type.upper() == 'SQ' else (R + L)
    
    # Construct risk period demand
    risk_paths = np.sum(weekly_paths[:, :tau], axis=1)
    mean_risk = float(np.mean(risk_paths))
    std_risk = float(np.std(risk_paths, ddof=1))
    
    simulator = InventorySimulator(lead_time=L, holding_cost=H_weekly, order_cost=K, shortage_cost=B)
    optimizer = GridSearchOptimizer(simulator, target_fill_rate=config.target_fill_rate)
    
    print(f"\n--- Optimizing {family} ({p_type}) ---")
    
    if p_type.upper() == 'SQ':
        # Continuous Review SQ
        q0 = calculate_eoq(K, np.mean(weekly_paths), H_weekly)
        q_grid = np.array([0.25, 0.50, 0.75, 1.0, 1.25, 1.50, 1.75, 2.0]) * q0
        ss_grid = np.arange(0, 4.01 * std_risk, 0.25 * std_risk)
        
        mean_ltd = np.mean(weekly_paths) * L
        
        opt_ss, opt_q, opt_s, opt_metrics = optimizer.optimize_sq(weekly_paths, ss_grid, q_grid, mean_ltd)
        print(f"  Optimal Safety Stock: {opt_ss:.2f}")
        print(f"  Optimal Reorder Point s: {opt_s:.2f}")
        print(f"  Optimal Order Quantity Q: {opt_q:.2f}")
        print(f"  Average Cost: {opt_metrics['total_cost']:.4f}")
        print(f"  Realized Fill Rate: {opt_metrics['fill_rate']:.4f}")
        
        optimization_results.append({
            'family': family, 'policy_type': '(s,Q)', 'safety_stock': opt_ss,
            'reorder_point': opt_s, 'order_quantity': opt_q, 'order_up_to_level': np.nan,
            'fill_rate': opt_metrics['fill_rate'], 'cycle_service_level': opt_metrics['cycle_service_level'],
            'total_cost': opt_metrics['total_cost'], 'avg_inventory': opt_metrics['avg_inventory'],
            'num_orders': opt_metrics['num_orders'], 'units_short': opt_metrics['units_short']
        })
    else:
        # Periodic Review RS
        ss_grid = np.arange(0, 4.01 * std_risk, 0.25 * std_risk)
        
        opt_ss, opt_S, opt_metrics = optimizer.optimize_rs(weekly_paths, ss_grid, mean_risk)
        print(f"  Optimal Safety Stock: {opt_ss:.2f}")
        print(f"  Optimal Order-Up-To S: {opt_S:.2f}")
        print(f"  Average Cost: {opt_metrics['total_cost']:.4f}")
        print(f"  Realized Fill Rate: {opt_metrics['fill_rate']:.4f}")
        
        optimization_results.append({
            'family': family, 'policy_type': '(R,S)', 'safety_stock': opt_ss,
            'reorder_point': np.nan, 'order_quantity': np.nan, 'order_up_to_level': opt_S,
            'fill_rate': opt_metrics['fill_rate'], 'cycle_service_level': opt_metrics['cycle_service_level'],
            'total_cost': opt_metrics['total_cost'], 'avg_inventory': opt_metrics['avg_inventory'],
            'num_orders': opt_metrics['num_orders'], 'units_short': opt_metrics['units_short']
        })


--- Optimizing GROCERY I (SQ) ---
  Optimal Safety Stock: 487622.56
  Optimal Reorder Point s: 1750781.15
  Optimal Order Quantity Q: 314433.58
  Average Cost: 543439.8849
  Realized Fill Rate: 0.9916

--- Optimizing BEVERAGES (SQ) ---
  Optimal Safety Stock: 588068.73
  Optimal Reorder Point s: 1565680.61
  Optimal Order Quantity Q: 260348.02
  Average Cost: 156552.7656
  Realized Fill Rate: 0.9966

--- Optimizing CLEANING (RS) ---
  Optimal Safety Stock: 0.00
  Optimal Order-Up-To S: 798371.82
  Average Cost: 1361.6536
  Realized Fill Rate: 1.0000


## 4. Optimization Summary Table

In [5]:
opt_df = pd.DataFrame(optimization_results)
create_policy_comparison_table(opt_df)

,family,policy_type,safety_stock,reorder_point,order_quantity,order_up_to_level,fill_rate,cycle_service_level,total_cost,avg_inventory,num_orders,units_short
0,GROCERY I,"(s,Q)",487622.56,1750781.15,314433.58,NaN,99.16%,93.13%,543439.88,609857.95,3.0,36107.05
1,BEVERAGES,"(s,Q)",588068.73,1565680.61,260348.02,NaN,99.66%,98.47%,156552.77,687134.05,3.0,12874.64
2,CLEANING,"(R,S)",0.00,NaN,NaN,798371.82,100.0%,100.0%,1361.65,446039.91,3.0,0.00
